# 📈 Notebook 4 — Risk Scoring & Business Insights
**Project:** Customer Churn Prediction  
**Author:** Aditya Bobade

---
### Objectives
- Assign a 0–100 churn risk score to every customer
- Segment into Low / Medium / High / Critical risk tiers
- Identify high-value customers at risk (revenue impact)
- Export risk-scored data for Power BI dashboard
- Generate actionable retention recommendations


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df_clean = pd.read_csv('../data/telco_churn_cleaned.csv')
df_bal   = pd.read_csv('../data/telco_churn_balanced.csv')

X = df_bal.drop(columns=['Churn']); y = df_bal['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
model.fit(X_train, y_train)

X_full = df_clean.drop(columns=['Churn']).reindex(columns=X_train.columns, fill_value=0)
probs  = model.predict_proba(X_full)[:,1]

df_clean['ChurnProbability'] = probs
df_clean['RiskScore']        = (probs * 100).round(1)
print("Risk scores assigned. Range:", df_clean['RiskScore'].min(), "–", df_clean['RiskScore'].max())


Risk scores assigned. Range: 1.6 – 91.3


In [2]:
# Risk Tier Segmentation
def tier(s):
    if s < 25:   return 'Low Risk'
    elif s < 50: return 'Medium Risk'
    elif s < 75: return 'High Risk'
    else:        return 'Critical Risk'

df_clean['RiskTier'] = df_clean['RiskScore'].apply(tier)
tier_order  = ['Low Risk','Medium Risk','High Risk','Critical Risk']
tier_colors = ['#2ecc71','#f39c12','#e74c3c','#8e44ad']

counts = [len(df_clean[df_clean['RiskTier']==t]) for t in tier_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
bars = axes[0].bar(tier_order, counts, color=tier_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Customers by Risk Tier', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Customers')
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, str(val),
                 ha='center', fontweight='bold')

axes[1].pie(counts, labels=tier_order, autopct='%1.1f%%',
            colors=tier_colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Risk Tier Distribution', fontsize=14, fontweight='bold')
plt.suptitle('Churn Risk Tier Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/11_risk_tier_distribution.png', bbox_inches='tight')
plt.show()


In [3]:
# Risk Score Histogram
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_clean['RiskScore'], bins=50, color='#3498db', edgecolor='white', alpha=0.85)
for thresh, col, lbl in [(25,'#2ecc71','Low'),(50,'#f39c12','Med'),(75,'#e74c3c','High')]:
    ax.axvline(thresh, color=col, lw=2, linestyle='--', label=f'{lbl} threshold')
ax.set_title('Churn Risk Score Distribution (0–100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Risk Score'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../charts/12_risk_score_distribution.png', bbox_inches='tight')
plt.show()


In [4]:
# Revenue at Risk
orig = pd.read_csv('../data/telco_churn.csv')
orig['RiskScore'] = df_clean['RiskScore'].values
orig['RiskTier']  = df_clean['RiskTier'].values

hi_risk      = orig[orig['RiskTier'].isin(['High Risk','Critical Risk'])]
rev_at_risk  = hi_risk['MonthlyCharges'].sum()
hv_at_risk   = hi_risk[hi_risk['MonthlyCharges'] > hi_risk['MonthlyCharges'].quantile(0.75)]

print(f"High+Critical Risk Customers: {len(hi_risk):,}")
print(f"Monthly Revenue at Risk     : ${rev_at_risk:,.2f}")
print(f"High-Value at Risk          : {len(hv_at_risk):,} customers (${hv_at_risk['MonthlyCharges'].sum():,.2f}/mo)")


High+Critical Risk Customers: 3,056
Monthly Revenue at Risk     : $229,890.66
High-Value at Risk          : 764 customers ($73,978.03/mo)


In [5]:
# Avg metrics by risk tier
avg = orig.groupby('RiskTier')[['tenure','MonthlyCharges','TotalCharges']].mean()
avg = avg.loc[tier_order]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['tenure','MonthlyCharges','TotalCharges']):
    bars = ax.bar(avg.index, avg[col], color=tier_colors, edgecolor='white')
    ax.set_title(f'Avg {col}', fontweight='bold')
    ax.set_xticklabels(avg.index, rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, avg[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.0f}', ha='center', fontsize=9, fontweight='bold')
plt.suptitle('Customer Metrics by Risk Tier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/13_metrics_by_risk_tier.png', bbox_inches='tight')
plt.show()


In [6]:
# Export for Power BI
orig.to_csv('../data/telco_churn_risk_scored.csv', index=False)
print("Saved: telco_churn_risk_scored.csv — ready for Power BI")

print("""
=== Retention Action Plan ===
┌─────────────────┬──────────────────────────────────────────────────────────┐
│ Risk Score      │ Recommended Action                                        │
├─────────────────┼──────────────────────────────────────────────────────────┤
│  0 – 25  Low    │ Standard engagement; quarterly check-in                   │
│ 25 – 50  Medium │ Loyalty rewards; offer contract upgrade incentive          │
│ 50 – 75  High   │ Proactive outreach; personalised retention offer           │
│ 75 – 100 Crit.  │ Immediate escalation; high-priority discount + callback    │
└─────────────────┴──────────────────────────────────────────────────────────┘
""")


Saved: telco_churn_risk_scored.csv — ready for Power BI

=== Retention Action Plan ===
┌─────────────────┬──────────────────────────────────────────────────────────┐
│ Risk Score      │ Recommended Action                                        │
├─────────────────┼──────────────────────────────────────────────────────────┤
│  0 – 25  Low    │ Standard engagement; quarterly check-in                   │
│ 25 – 50  Medium │ Loyalty rewards; offer contract upgrade incentive          │
│ 50 – 75  High   │ Proactive outreach; personalised retention offer           │
│ 75 – 100 Crit.  │ Immediate escalation; high-priority discount + callback    │
└─────────────────┴──────────────────────────────────────────────────────────┘

